# ChessFM — Kaggle GPU T4 x2 Training, Evaluation & GGUF Export

End-to-end resumable training notebook tailored for **Kaggle GPU T4 x2 (30 hrs/week free)** and 12-hour session limits.

Supported tracks:
- **`v0`** (`Qwen/Qwen2.5-3B-Instruct`): Baseline RL/SFT reasoning track
- **`v1`** (`Qwen/Qwen3-4B`): Upgraded research & RL track
- **`v2`** (`unsloth/gemma-4-E4B-it`): Engine-grounded AI Chess Coach track (ungated)

## 1. Environment Setup
Install Stockfish (`/usr/games/stockfish`), clone the ChessFM repository, and install GPU dependencies (`pip install -e .[gpu]`).

In [ ]:
!apt-get -qq update && apt-get -qq install -y stockfish
!test -d /kaggle/working/chess-fm || git clone https://github.com/ArmaanSethi/chess-fm.git /kaggle/working/chess-fm
%cd /kaggle/working/chess-fm
!pip install -q -e ".[gpu]"
!nvidia-smi && /usr/games/stockfish --version || true

## 2. Training Configuration
Select the model track (`v0`, `v1`, or `v2`), prompt ASCII board toggle, GRPO stage (`0` = Stockfish-free legality/format, `1` = Stockfish move quality, `2` = full shaped reward), and step budget.

In [ ]:
import os
from pathlib import Path

TRACK = "v2"  # "v0" | "v1" | "v2"
INCLUDE_ASCII = True  # Include ASCII board rendering in prompts
GRPO_STAGE = 0  # 0 | 1 | 2
STEPS = 1000  # Resumable GRPO step budget
SFT_EPOCHS = 3
EVAL_GAMES = 20
STOCKFISH_LEVEL = 1

TRACK_CONFIG = {
    "v0": {"model": "Qwen/Qwen2.5-3B-Instruct", "family": "qwen25"},
    "v1": {"model": "Qwen/Qwen3-4B", "family": "qwen3"},
    "v2": {"model": "unsloth/gemma-4-E4B-it", "family": "gemma4"},
}
assert TRACK in TRACK_CONFIG, f"Unsupported TRACK={TRACK!r}"

BASE_MODEL = TRACK_CONFIG[TRACK]["model"]
MODEL_FAMILY = TRACK_CONFIG[TRACK]["family"]
WORK_ROOT = Path("/kaggle/working/checkpoints") / TRACK
SFT_DIR = WORK_ROOT / "sft"
SFT_MERGED = SFT_DIR / "final_merged"
GRPO_DIR = WORK_ROOT / f"grpo_stage{GRPO_STAGE}"
GRPO_FINAL = GRPO_DIR / f"stage{GRPO_STAGE}_final"
EXPORT_DIR = Path("/kaggle/working/exports") / f"chessfm-{TRACK}-{MODEL_FAMILY}"
DATA_PATH = "data_generation/sft_train.jsonl"

os.environ["PATH"] = "/usr/games:" + os.environ.get("PATH", "")
ASCII_FLAG = "--include-ascii" if INCLUDE_ASCII else ""

print(f"Track={TRACK} | Model={BASE_MODEL} | Family={MODEL_FAMILY}")
print(f"SFT_MERGED={SFT_MERGED}")
print(f"GRPO_FINAL={GRPO_FINAL}")

## 3. SFT + Merged Checkpoint
Runs LoRA SFT via `scripts/kaggle_train.py --task sft` and merges weights into `final_merged` for downstream GRPO or GGUF export.

In [ ]:
sft_cmd = (
    f"python scripts/kaggle_train.py --task sft "
    f"--model {BASE_MODEL} "
    f"--data {DATA_PATH} "
    f"--output {SFT_DIR} "
    f"--epochs {SFT_EPOCHS} "
    f"--merge-and-save {ASCII_FLAG}"
).strip()
print(sft_cmd)
!{sft_cmd}

## 4. Resumable GRPO RL Training
Starts from the merged SFT checkpoint and saves periodic checkpoints so runs resume cleanly across Kaggle 12-hour sessions with `--resume`.

In [ ]:
grpo_model = str(SFT_MERGED) if SFT_MERGED.exists() else BASE_MODEL
grpo_cmd = (
    f"python scripts/kaggle_train.py --task grpo "
    f"--model {grpo_model} "
    f"--stage {GRPO_STAGE} "
    f"--steps {STEPS} "
    f"--output {GRPO_DIR} "
    f"--checkpoint-every 100 "
    f"--resume {ASCII_FLAG}"
).strip()
print(grpo_cmd)
!{grpo_cmd}

## 5. Stockfish Match Eval & 4-Tier Evaluation Suite
Evaluates the trained checkpoint in color-balanced games against Stockfish and runs the 4-Tier evaluation harness (`chessfm.eval.suite`).

In [ ]:
eval_model = str(GRPO_FINAL) if GRPO_FINAL.exists() else grpo_model
eval_cmd = (
    f"python scripts/kaggle_train.py --task eval "
    f"--model {eval_model} "
    f"--games {EVAL_GAMES} "
    f"--stockfish-level {STOCKFISH_LEVEL}"
)
print(eval_cmd)
!{eval_cmd}
!python -m chessfm.eval.suite --demo

## 6. GGUF + Ollama Modelfile Export
Generates a quantized GGUF export plan and ready-to-run Ollama `Modelfile` with family-specific stop tokens (`gemma4`, `qwen3`, or `qwen25`).

In [ ]:
export_model = str(SFT_MERGED) if SFT_MERGED.exists() else eval_model
export_cmd = (
    f"python scripts/export_gguf.py "
    f"--model {export_model} "
    f"--output {EXPORT_DIR} "
    f"--quant q4_k_m "
    f"--family {MODEL_FAMILY}"
)
print(export_cmd)
!{export_cmd}